# M09A — Model Selection & Generalisation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tulip-lab/pattern-classification-lab/blob/develop/M09-Model-Evaluation/M09A-Model-Selection-and-Generalisation.ipynb)

**Pattern Classification practical · approximately 2 hours · M09 · CLO4**

This is the canonical student-facing practical for session `M09A`.
It was newly authored for the Pattern Classification Lab using concepts and
public migration references from `tulip-lab/flip01`: `F1B-00-MLPipeline.ipynb`, `F1B-01-MLGridSearch.ipynb`, `F1B-02-MLValidationCurves.ipynb`. No legacy
notebook cells or outputs are copied verbatim. The implementation uses current
public APIs and synthetic or scikit-learn-bundled data.

**Licence:** code cells are available under MPL-2.0; original narrative teaching
content is available under CC BY-NC-SA 4.0. Third-party works and datasets retain
their own terms. See the repository `LICENSE`, `CONTENT-LICENSE.md`, and
`NOTICE.md`.

## Overview

A high training score is not evidence of generalisation. You will protect a final test set, tune a pipeline only within the training data, and use validation and learning curves to diagnose capacity and data limitations.

Work through the guided cells first, then complete the challenge in your own
copy. Keep your interpretation beside the code: a result without an argument
is not complete evidence.

## Learning Objectives

- Separate model selection from final performance estimation.
- Tune a complete pipeline using stratified cross-validation.
- Interpret validation and learning curves as bias/variance evidence.
- Report uncertainty and limitations alongside a held-out score.

## Expected Output and Evidence

Retain the following evidence in your executed notebook:

- a cross-validated pipeline search with protected test data
- validation and learning curves paired with uncertainty evidence
- a model-comparison recommendation that states limitations

Do not include credentials, private data, or another learner's work.

## Prerequisites

- Classification metrics
- Cross-validation
- Pipelines and hyperparameters
- Basic interpretation of means and standard deviations

## Session Plan

| Time | Activity | Evidence |
|---:|---|---|
| 0–10 min | Orientation and diagnostic prompt | Initial prediction or sketch |
| 10–30 min | Background concepts and setup | Concept notes and reproducible environment |
| 30–75 min | Guided practice | Executed analysis with interpretations |
| 75–105 min | Student challenge | Independent model choice and evidence |
| 105–115 min | Testing and debugging | Passing checks and resolved issues |
| 115–120 min | Submission and reflection | Concise evidence-based reflection |

## Background Concepts

Model selection estimates which workflow and hyperparameters are promising; final assessment estimates performance after selection. Reusing the test set for choices makes its score optimistic.

A validation curve changes one complexity control. A learning curve changes training-set size. Together, the training/validation gap and their absolute levels help distinguish high bias, high variance, and data scarcity. Cross-validation spread is useful uncertainty evidence, but it is not a guarantee about deployment shift.

## Accessibility and Responsible Use

Every visual result in the guided path is paired with a printed numeric summary. Do not rely on colour alone in your challenge evidence: add labels, line styles, markers, or a compact table as appropriate. Add concise alt text when exporting figures for another format. Use only public, approved, or synthetic data.

Follow the collaboration, acknowledgement, and AI-use rules published by your current offering; this institution-neutral practical does not define those rules.

## Setup

Run the next cell before starting. All datasets are bundled with scikit-learn or generated locally; no download is required.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.metrics import balanced_accuracy_score, classification_report
from sklearn.model_selection import (
    GridSearchCV, StratifiedKFold, learning_curve, train_test_split, validation_curve
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

RANDOM_STATE = 42
plt.style.use("seaborn-v0_8-whitegrid")

## Guided Practice

Follow the sequence and pause at each interpretation prompt before moving on.

### 1. Protect a final test set

Do not use `X_test` or `y_test` until the search has finished.

In [ ]:
dataset = load_breast_cancer()
X_train, X_test, y_train, y_test = train_test_split(
    dataset.data, dataset.target, test_size=0.25,
    stratify=dataset.target, random_state=RANDOM_STATE,
)
print("training samples:", len(X_train), "protected test samples:", len(X_test))
print("training class proportions:", np.bincount(y_train) / len(y_train))

### 2. Tune the entire workflow

The scaler is refitted inside each fold. Balanced accuracy gives each class equal importance.

In [ ]:
pipeline = Pipeline([
    ("scale", StandardScaler()),
    ("svm", SVC(kernel="rbf")),
])
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
parameter_grid = {
    "svm__C": [0.1, 1, 10],
    "svm__gamma": ["scale", 0.01, 0.1],
}
search = GridSearchCV(
    pipeline, parameter_grid, scoring="balanced_accuracy", cv=cv,
    return_train_score=True, n_jobs=1,
)
search.fit(X_train, y_train)
print("best parameters:", search.best_params_)
print(f"best mean CV balanced accuracy: {search.best_score_:.3f}")

### 3. Inspect a validation curve

Vary C while keeping the selected gamma fixed. The shaded bands show fold-to-fold variation, not confidence intervals.

In [ ]:
c_values = np.logspace(-2, 2, 7)
curve_model = Pipeline([
    ("scale", StandardScaler()),
    ("svm", SVC(kernel="rbf", gamma=search.best_params_["svm__gamma"])),
])
train_scores, validation_scores = validation_curve(
    curve_model, X_train, y_train, param_name="svm__C", param_range=c_values,
    cv=cv, scoring="balanced_accuracy", n_jobs=1,
)
for values, label in [(train_scores, "training"), (validation_scores, "validation")]:
    mean, std = values.mean(axis=1), values.std(axis=1)
    plt.semilogx(c_values, mean, marker="o", label=label)
    plt.fill_between(c_values, mean - std, mean + std, alpha=0.15)
plt.xlabel("C")
plt.ylabel("balanced accuracy")
plt.title("Validation curve")
plt.legend()
plt.show()

### 4. Inspect a learning curve

If validation performance is still rising at the right edge, more representative data may help.

In [ ]:
sizes, train_learning, validation_learning = learning_curve(
    search.best_estimator_, X_train, y_train,
    train_sizes=np.linspace(0.2, 1.0, 5), cv=cv,
    scoring="balanced_accuracy", shuffle=True, random_state=RANDOM_STATE, n_jobs=1,
)
plt.plot(sizes, train_learning.mean(axis=1), marker="o", label="training")
plt.plot(sizes, validation_learning.mean(axis=1), marker="o", label="validation")
plt.fill_between(sizes, validation_learning.mean(axis=1) - validation_learning.std(axis=1),
                 validation_learning.mean(axis=1) + validation_learning.std(axis=1), alpha=0.15)
plt.xlabel("training samples")
plt.ylabel("balanced accuracy")
plt.title("Learning curve")
plt.legend()
plt.show()

### 5. Evaluate once on the protected test set

This score estimates the performance of the entire selection procedure more honestly than the best training score.

In [ ]:
final_predictions = search.best_estimator_.predict(X_test)
final_balanced_accuracy = balanced_accuracy_score(y_test, final_predictions)
print(f"final held-out balanced accuracy: {final_balanced_accuracy:.3f}")
print(classification_report(y_test, final_predictions, target_names=dataset.target_names))

## Student Challenge

Compare the selected RBF SVM with a linear SVM under the same folds and balanced-accuracy metric. Report paired fold scores, the mean difference, and whether the practical gain justifies the extra flexibility. Keep the protected test set out of this comparison.

In [ ]:
from sklearn.model_selection import cross_val_score

linear_candidate = Pipeline([
    ("scale", StandardScaler()),
    ("svm", SVC(kernel="linear", C=1.0)),
])

# TODO: calculate fold scores for `linear_candidate` and `search.best_estimator_`
# using the same `cv`, then compare the paired score differences.
paired_differences = np.array([])
paired_differences

### Challenge success criteria

- Candidates use the same data splits and metric.
- The comparison reports fold-level paired differences, not only means.
- No further choice is made from protected test performance.
- The recommendation includes uncertainty and model-complexity considerations.

## Testing and Debugging

Run these checks after the guided practice. If one fails, inspect shapes, label order, random seeds, and whether preprocessing was fitted only on training data.

In [ ]:
# Exercise invalid-configuration and missing-information behaviour.
try:
    pipeline.set_params(svm__not_a_parameter=1)
except ValueError as error:
    print("Expected invalid-parameter failure:", str(error).split("\n")[0])
else:
    raise AssertionError("An unknown pipeline parameter should fail clearly.")

try:
    search.best_estimator_.predict(np.full((1, X_train.shape[1]), np.nan))
except ValueError as error:
    print("Expected missing-value failure:", str(error).split("\n")[0])
else:
    raise AssertionError("The selected workflow should reject NaN input.")

In [ ]:
assert set(search.best_params_) == {"svm__C", "svm__gamma"}
assert len(search.cv_results_["mean_test_score"]) == 9
assert train_scores.shape == validation_scores.shape == (len(c_values), cv.get_n_splits())
assert sizes[-1] <= len(X_train)
assert 0.0 <= final_balanced_accuracy <= 1.0
print("M09A guided-practice checks passed.")

## Extension Task

Use nested cross-validation to estimate the full tuning procedure without a single fixed test split. Compare its distribution with the one held-out score and discuss computational cost.

## Submission and Reflection

Submit your completed notebook (or an HTML export) with outputs visible. Include: 

1. the challenge code and result;
2. one figure or compact results table;
3. a 150–250 word reflection answering: **What changed, why did it change, and what evidence supports your explanation?**

Follow your offering's published collaboration, acknowledgement, and AI-use rules. You remain responsible for checking and explaining all submitted work.

## References

- Sources: `F1B-00-MLPipeline.ipynb`, `F1B-01-MLGridSearch.ipynb`, and `F1B-02-MLValidationCurves.ipynb`.
- scikit-learn User Guide: [Tuning the hyper-parameters of an estimator](https://scikit-learn.org/stable/modules/grid_search.html)
- scikit-learn User Guide: [Learning curves](https://scikit-learn.org/stable/modules/learning_curve.html)